# 🚀 SpecSense AI: Step-by-Step Requirements Intelligence Pipeline
This notebook breaks down the machine learning and rule-based pipeline into modular, easy-to-understand steps.

### Pipeline Architecture:
1. **Step 1:** Imports & Environment Setup
2. **Step 2:** Synthetic Dataset Generation & Inspection
3. **Step 3:** Text Feature Extraction (TF-IDF Vectorization)
4. **Step 4:** Defect Classification Model Training (Logistic Regression)
5. **Step 5:** Rule-Based Heuristic Explainability & Dynamic Rewrites
6. **Step 6:** Model Persistence (`.joblib` Export for FastAPI Backend)
7. **Step 7:** Interactive Playground (Sentence Scoring & Analysis)


## Step 1: Imports & Dependencies

In [1]:
import os
import sys
import importlib.util
import joblib
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score

print('✓ All required ML packages imported successfully!')


✓ All required ML packages imported successfully!


## Step 2: Requirement Dataset

In [2]:
dataset_path = os.path.abspath(os.path.join('..', 'backend', 'data', 'requirements_dataset.csv'))
generator_path = os.path.abspath(os.path.join('..', 'backend', 'data', 'dataset_generator.py'))

if not os.path.exists(dataset_path):
    print('Generating dataset from backend/data/dataset_generator.py...')
    spec = importlib.util.spec_from_file_location('dataset_generator', generator_path)
    generator = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(generator)
    df = generator.generate_srs_dataset()
    os.makedirs(os.path.dirname(dataset_path), exist_ok=True)
    df.to_csv(dataset_path, index=False)
else:
    df = pd.read_csv(dataset_path)

df.head(10)
print(f'Loaded dataset with {len(df)} rows from {dataset_path}')

Loaded dataset with 11603 rows from d:\Aarya Manandhar\Specai\backend\data\requirements_dataset.csv


## Step 3: TF-IDF Text Feature Extraction

In [3]:
vectorizer = TfidfVectorizer(ngram_range=(1, 3), max_df=0.85, stop_words='english', lowercase=True)
X_vec = vectorizer.fit_transform(df['requirement_text'].astype(str))
print('Feature Matrix Shape:', X_vec.shape)
print('Learned Vocabulary (sample):', sorted(list(vectorizer.vocabulary_.keys()))[:12])

Feature Matrix Shape: (11603, 33180)
Learned Vocabulary (sample): ['00', '00 00', '00 00 utc', '00 utc', '000', '000 000', '000 000 active', '000 000 applications', '000 000 articles', '000 000 connected', '000 000 evaluations', '000 000 messages']


## Step 4: Model Training

In [4]:
categories = ['is_ambiguous', 'is_incomplete', 'is_untestable', 'is_weak']
models = {}
thresholds = {}

X_train, X_val, y_train, y_val = train_test_split(
    X_vec,
    df[categories],
    test_size=0.2,
    random_state=42,
    shuffle=True
)

for category in categories:
    y_train_cat = y_train[category].astype(int)
    y_val_cat = y_val[category].astype(int)

    if len(np.unique(y_train_cat)) < 2:
        print(f"Skipping training for {category}: only one class present in training data.")
        continue

    n_splits = min(5, y_train_cat.sum(), len(y_train_cat) - y_train_cat.sum())
    if n_splits < 2:
        n_splits = 2

    clf = LogisticRegressionCV(
        Cs=[0.01, 0.1, 1.0, 10.0],
        cv=StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42),
        penalty='l2',
        solver='liblinear',
        class_weight='balanced',
        max_iter=500,
        scoring='f1',
        n_jobs=-1,
        refit=True,
        verbose=0
    )
    clf.fit(X_train, y_train_cat)
    models[category] = clf

    y_val_probs = clf.predict_proba(X_val)[:, 1]
    best_threshold = 0.5
    best_f1 = 0.0
    for threshold in np.linspace(0.1, 0.9, 41):
        y_pred_val = (y_val_probs >= threshold).astype(int)
        f1 = precision_recall_fscore_support(y_val_cat, y_pred_val, average='binary', zero_division=0)[2]
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold
    thresholds[category] = best_threshold

    y_pred_val = (y_val_probs >= best_threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(y_val_cat, y_pred_val, average='binary', zero_division=0)
    roc_auc = roc_auc_score(y_val_cat, y_val_probs) if len(np.unique(y_val_cat)) > 1 else float('nan')

    print(f'✓ Trained {category}: threshold={best_threshold:.2f}, precision={precision:.2f}, recall={recall:.2f}, f1={f1:.2f}, auc={roc_auc:.2f}')


✓ Trained is_ambiguous: threshold=0.58, precision=1.00, recall=1.00, f1=1.00, auc=1.00
✓ Trained is_incomplete: threshold=0.58, precision=0.98, recall=0.96, f1=0.97, auc=0.99
✓ Trained is_untestable: threshold=0.32, precision=1.00, recall=1.00, f1=1.00, auc=1.00
✓ Trained is_weak: threshold=0.40, precision=1.00, recall=1.00, f1=1.00, auc=1.00


In [5]:
# Step 4b: Evaluate validation performance across all categories
from sklearn.metrics import classification_report

for category in categories:
    y_val_cat = y_val[category].astype(int)
    y_pred_val = models[category].predict(X_val)
    print(f"\nValidation report for {category}:")
    print(classification_report(y_val_cat, y_pred_val, zero_division=0))

# Show a few misclassified examples for the first category
first_cat = categories[0]
mis_idx = (y_val[first_cat].astype(int) != models[first_cat].predict(X_val))
if mis_idx.any():
    print(f"\nSample misclassified rows for {first_cat}:")
    sample = y_val[mis_idx].index[:5]
    print(df.loc[sample, ['requirement_text', first_cat]])
else:
    print(f"\nNo misclassifications found for {first_cat} on validation set.")


Validation report for is_ambiguous:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1702
           1       1.00      1.00      1.00       619

    accuracy                           1.00      2321
   macro avg       1.00      1.00      1.00      2321
weighted avg       1.00      1.00      1.00      2321


Validation report for is_incomplete:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1920
           1       0.95      0.98      0.96       401

    accuracy                           0.99      2321
   macro avg       0.97      0.98      0.98      2321
weighted avg       0.99      0.99      0.99      2321


Validation report for is_untestable:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1745
           1       1.00      0.99      1.00       576

    accuracy                           1.00      2321
   macro avg   

## Step 5: Save Model Artifacts for Backend

In [6]:
model_dir = os.path.abspath(os.path.join('..', 'backend', 'app', 'ml', 'models'))
os.makedirs(model_dir, exist_ok=True)

joblib.dump(vectorizer, os.path.join(model_dir, 'vectorizer.joblib'))
for category, clf in models.items():
    joblib.dump(clf, os.path.join(model_dir, f'{category}_model.joblib'))

print(f'✓ Saved vectorizer and classifier binaries to: {model_dir}')

✓ Saved vectorizer and classifier binaries to: d:\Aarya Manandhar\Specai\backend\app\ml\models


## Step 6: Test Prediction Engine

In [7]:
sample_sentence = 'The application should update user records periodically.'
sample_vec = vectorizer.transform([sample_sentence])

print(f"Analyzing sentence: '{sample_sentence}'")
for cat, model in models.items():
    prob = model.predict_proba(sample_vec)[0][1]
    threshold = thresholds.get(cat, 0.5)
    label = 'defect' if prob >= threshold else 'pass'
    print(f" - {cat}: Probability = {prob:.2%}, threshold = {threshold:.2f}, label = {label}")


Analyzing sentence: 'The application should update user records periodically.'
 - is_ambiguous: Probability = 86.28%, threshold = 0.58, label = defect
 - is_incomplete: Probability = 0.85%, threshold = 0.58, label = pass
 - is_untestable: Probability = 11.70%, threshold = 0.32, label = pass
 - is_weak: Probability = 22.90%, threshold = 0.40, label = pass
